# DI-1 — Document Forgery Detection

Detect whether a document has been forged or contains tampered content (altered text, manipulated stamps, edited signatures, etc.).

**Model:** `DI-1`  
**Input:** Image files (`.jpg`, `.jpeg`, `.png`)

---
### Contents
1. Setup
2. One-call detection
3. Two-step: upload now, poll later
4. Visualize results — heatmap
5. Error handling
6. Async client

---
## 1. Setup

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
from authenta.authenta_client import AuthentaClient
from authenta import (
    AuthentaError,
    AuthenticationError,
    QuotaExceededError,
    InsufficientCreditsError,
)

API_KEY  = "API_KEY_HERE"
BASE_URL = "https://platform.authenta.ai"

client = AuthentaClient(
    base_url=BASE_URL,
    api_key=API_KEY,
)

print("Client ready.")

---
## 2. One-call Detection

`client.process()` uploads the document and blocks until the result is ready.

The result contains:

| Field | Type | Description |
| :-- | :-- | :-- |
| `isFake` | `bool` | `True` if the document is detected as forged |
| `isTampered` | `bool` | `True` if any region of the document shows signs of tampering |

In [ ]:
DOCUMENT = "../data_samples/document-intelligence/Agreement.jpg"

In [ ]:
media = client.process(DOCUMENT, model_type="DI-1")

result = client.get_result(media)

print(f"Job ID     : {media['id']}")
print(f"Status     : {media['status']}")
print(f"Is Fake    : {result['isFake']}")
print(f"Is Tampered: {result['isTampered']}")

---
## 3. Two-step: Upload Now, Poll Later

Use `upload_file()` to start the job, then call `wait_for_media()` when you're ready to collect the result.

In [ ]:
# Step 1 — upload
upload_meta = client.upload_file(DOCUMENT, model_type="DI-1")
jobid = upload_meta["job"]["id"]
print(f"Uploaded. Job ID : {jobid}")
print(f"          Response : {upload_meta}")

In [ ]:
# Step 2 — poll until done
media = client.wait_for_media(jobid, interval=5.0, timeout=300.0)
result = client.get_result(media)

print(f"Status     : {media['status']}")
print(f"Is Fake    : {result['isFake']}")
print(f"Is Tampered: {result['isTampered']}")

---
## 4. Visualize Results — Heatmap

Generate a heatmap overlay highlighting regions of the document that are flagged as tampered or forged.

In [ ]:
from authenta.visualization import save_heatmap

media = client.process(DOCUMENT, model_type="DI-1")

os.makedirs("results", exist_ok=True)
_ = save_heatmap(
    media=media,
    out_path="results"
)
print("Saved: results/")

In [ ]:
from IPython.display import Image as IPImage

heatmap_file = f"results/{media['id']}.png"
IPImage(heatmap_file, width=700)

---
## 5. Error Handling

In [ ]:
try:
    media = client.process(DOCUMENT, model_type="DI-1")
    result = client.get_result(media)
    print(f"Status     : {media['status']}")
    print(f"Is Fake    : {result['isFake']}")
    print(f"Is Tampered: {result['isTampered']}")
except AuthenticationError:
    print("Authentication failed — check your API key.")
except QuotaExceededError:
    print("API quota exceeded — upgrade your plan.")
except InsufficientCreditsError:
    print("Not enough credits.")
except TimeoutError as e:
    print(f"Timed out: {e}")
except AuthentaError as e:
    print(f"API error [{e.code}]: {e.message}")

---
## 6. Async Client

In [ ]:
import asyncio
from authenta.async_authenta_client import AsyncAuthentaClient

In [ ]:
# One-call — async
async def detect_single():
    async with AsyncAuthentaClient(
        base_url="https://platform.authenta.ai",
        api_key="API_KEY_HERE",
    ) as client:
        media = await client.process(DOCUMENT, model_type="DI-1")
        result = client.get_result(media)
        print(f"Status     : {media['status']}")
        print(f"Is Fake    : {result['isFake']}")
        print(f"Is Tampered: {result['isTampered']}")

await detect_single()

In [ ]:
# Two-step — async
async def detect_two_step():
    async with AsyncAuthentaClient(
        base_url="https://platform.authenta.ai",
        api_key="API_KEY_HERE",
    ) as client:
        upload_meta = await client.upload_file(DOCUMENT, model_type="DI-1")
        jobid = upload_meta["job"]["id"]
        print(f"Uploaded: {jobid}")

        media = await client.wait_for_media(jobid)
        result = client.get_result(media)
        print(f"Status     : {media['status']}")
        print(f"Is Fake    : {result['isFake']}")
        print(f"Is Tampered: {result['isTampered']}")

await detect_two_step()

In [ ]:
# Batch processing multiple documents (async)
async def process_batch(doc_paths: list):
    async with AsyncAuthentaClient(
        base_url="https://platform.authenta.ai",
        api_key="API_KEY_HERE",
    ) as client:
        tasks = [client.process(p, model_type="DI-1") for p in doc_paths]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        for path, res in zip(doc_paths, results):
            if isinstance(res, Exception):
                print(f"[FAILED] {path}: {res}")
            else:
                r = client.get_result(res)
                print(f"[OK] {path}: isFake={r['isFake']}, isTampered={r['isTampered']}")

await process_batch([DOCUMENT])